# NSP Intent Configuration System -- End-to-End Walkthrough
### Automated Nokia NSP Intent JSON Generation via Fine-tuned Qwen3.5-9B

**In one sentence:** translate a natural-language network service request into a ready-to-deploy Nokia NSP Intent JSON.

**Final metrics:** 330/330 test samples pass all four validation tiers (JSON validity, intent-type accuracy, field recall, value accuracy — all 100%). The 11 hand-crafted Golden Tests also pass 100% in the final golden-only regression run.

---

### How this notebook is organized

1. **Problem background** -- why is NSP Intent JSON hard to write by hand?
2. **Methodology timeline** -- the five milestones (M1 -> M3.5) and key decisions
3. **Core contribution** -- the pure-function generator contract (eliminates training-data information leakage)
4. **Validator stack** -- four-tier YANG-driven validation, demonstrated with deliberate failure cases
5. **End-to-end demos** -- full inference pipeline on all 9 intent types
6. **Overall evaluation** -- full test-set metrics + cross-milestone comparison
7. **Conclusions and future work**

## 1. Problem Background -- Why is NSP Intent hard to write?

Nokia NSP (Network Services Platform) is widely used by telecom operators. Its core abstraction is the **Intent** -- a declarative JSON object describing the desired state of a network service.

### A concrete example

An operator might express their need in plain English:

> *"Set up a VPRN L3 VPN for customer 5, named VPRN-100-DataCenter, on device 192.168.0.16, > service ID 100, with two interfaces: GPU-Cluster-Compute on 10.100.1.1/24, > GPU-Cluster-Storage on 10.100.2.1/24."*

Converting that sentence into a fully-qualified, offline-validated NSP intent JSON requires:

| Dimension | Complexity |
|---|---|
| **Field count** | A typical multi-site VPRN has 30-50 user-visible fields; after default merging the nested JSON exceeds 400 lines |
| **Path depth** | YANG paths reach 5-6 levels deep, e.g. `site-details.site[0].interface-details.interface[0].sap.port-id` |
| **Type constraints** | Every field has strict type rules (int32/string/enum), ranges (e.g. `vlan in 1..4094`), regex patterns, and length limits |
| **Cross-field semantics** | Invariants like SDP bidirectionality, distinct site device IDs, E-Tree leaf isolation cannot be expressed in a single field |

### What a hand-written VPRN JSON actually looks like (excerpt)

```json
{
  "nsp-service-intent:intent": [{
    "intent-specific-data": {
      "vprn:vprn": {
        "site-details": {
          "site": [{
            "interface-details": {
              "interface": [{
                "sap": { "port-id": "1/2/c4/1", /* ...8 more fields... */ },
                "ipv4": { "primary": { "address": "10.100.1.1", "prefix-length": 24 },
                           "bfd": {}, "neighbor-discovery": { "host-route": {}, "limit": {} },
                           "icmp": { "redirects": {}, "unreachables": {} }, ... },
                "vpls": { "evpn": { "arp": { /* 5 fields */ } }, ... },
                "hold-time": { "ipv4": {...}, "ipv6": {...} }, ...
              }, /* second interface */ {...} ]
            },
            "auto-bind-tunnel": { "resolution-filter": { "bgp": true }, "resolution": "filter" },
            "bgp-vpn-backup": {}, "ip-transports": {}, "confederation": {}, ...
          }]
        }
      }
    },
    "intent-type": "vprn", "intent-type-version": "2", "olc-state": "deployed",
    "service-name": "VPRN-100-DataCenter", "template-name": "VPRNServiceTemplate"
  }]
}
```

Any typo, type mismatch, or semantic conflict (e.g. SDP not bidirectional) causes deployment to fail. Engineers must simultaneously know **YANG schema structure + NSP API conventions + service-specific network semantics** -- the entry bar is high.

### What this project delivers

**Input:** one natural-language sentence -> **Output:** NSP RESTCONF-shaped Intent JSON that passes offline schema and semantic validation. Live NSP acceptance is future integration work.

## 2. Methodology Timeline -- M1 through M3.5

The project evolved through five iteration milestones, each with its own technical decisions and problem-solving process. This table lists the headlines; full stories live in `docs/technical_report.md`.

| Milestone | Focus | Key challenges & solutions | Outcome |
|---|---|---|---|
| **M1** | YANG validator MVP (Tier 1+2) | Use pyang to parse Nokia YANG modules, handle `import` / `uses grouping` / `typedef` chains | Baseline on 3 intent types |
| **M2** | Replace hand-written path maps with YANG-driven resolution | Swap `VPRN_SITE_SKELETON` etc. for `yang_schema.resolve_path()`; 307/307 samples byte-identical before/after | Removes tech debt |
| **M2.5** | Inference quality fix | **Root cause:** Qwen3.5 chat template leaves `<think>` empty-and-closed at training but open at inference — the model has never seen its own open-`<think>` context, so it hallucinates JS-like output. **Fix:** `enable_thinking=False` so the inference prompt ends byte-identically to the training prompt | Parseability recovers fully |
| **M3** | Scale from 3 to 9 intent types | Add vpls/ies/etree/cpipe/evpn-epipe/evpn-vpls. Fix eval-whitelist hard-coding (which silently skipped new types); bump `max_length` from 2048 -> 8192 (etree samples reach ~2700 tokens) | 98.8% on test set |
| **M3.5** | Pure-function generator contract + E-Tree SDP fix + Nokia canonical-payload mining | See sections 3 and 4 | Test set & Golden **100%** |

### System architecture at a glance

```
  Natural-language instruction
       |
       v
  +-----------------------+
  | Qwen3.5-9B + LoRA     |   (fixed SYSTEM_PROMPT, enable_thinking=False)
  +-----------------------+
       |
       v
  {intent_type, template_name, fill_values}   <-- flat dot-path dict
       |
       v
  +-----------------------+
  | merge_fill_values()   |   (YANG-driven expansion to nested JSON)
  +-----------------------+
       |
       v
  NSP RESTCONF-shaped JSON (offline validated)  -------+
       |                     |
       v                     v
  +------------------------------------------------+
  | Validation:                                    |
  |   Tier 1+2  YANG path / type / range / enum    |
  |   Tier 3    merged-structure integrity         |
  |   Tier 4    cross-field semantic rules         |
  |   Tier 6    Nokia canonical similarity (warn)  |
  +------------------------------------------------+
```

## 3. Core Contribution -- The Pure-Function Generator Contract

This is the single most important methodological decision of M3.5 -- it lifted EVPN value accuracy from 73-78% up to **100%**.

### The problem -- information leakage in synthetic training data

Template-based training-data synthesis has a subtle trap: **any value sampled randomly inside the generator that never appears in the instruction template is unpredictable by the model, no matter how many epochs you train for.**

The M3.5 baseline evpn-epipe generator contained code like this:
```python
# BAD: random values never flow into the instruction
def generate_evpn_epipe_values(...):
    rd = f"{random.randint(1,65535)}:{random.randint(1,65535)}"   # random
    vni = random.randint(1000, 16000000)                           # random
    eth_tag = random.randint(1, 4094)                              # random
    ...
```

The instruction template only mentions `service_name`, `customer_id`, etc., yet the training target contains values like `rd="31729:8842"`, `vni=9485123`, `eth_tag=3071` that the model has no way to see -- no amount of training can raise accuracy on those fields above the random baseline.

### The fix -- a strict pure-function contract

> Every value returned by a generator must be one of:
>
> 1. A direct copy of an instruction-visible argument, OR
> 2. A fixed constant for this intent type (e.g. `mtu=1500`), OR
> 3. A deterministic function of instruction-visible arguments (e.g. `RD = "65000:{ne_service_id}"`)

Rewritten under that contract:

```python
# GOOD: pure function from visible args to output
def generate_evpn_epipe_values(*, service_name, customer_id, ne_service_id,
                                evi, evpn_type, vlan, device, port,
                                local_ac, remote_ac):
    rd = f"65000:{ne_service_id}"        # deterministic: fixed ASN + service ID
    return {
        "service-name": service_name,    # direct from arg
        "mtu": 1500,                     # constant
        "description": f"{service_name} EVPN service",  # deterministic derivation
        "site-a.local-ac.eth-tag": vlan,                 # = access VLAN
        "site-a.mpls.bgp-instance.route-distinguisher": rd,
        ...
    }
```

Derivation rules chosen to match real Nokia operator practice:

| Field | Rule | Notes |
|---|---|---|
| RD / RT | `"65000:{ne_service_id}"` | Fixed ASN 65000 + service ID |
| VNI | `ne_service_id` | VXLAN ID equals service ID |
| vsi-import | `["{service_name}-import"]` | Derived from service name |
| vsi-export | `["{service_name}-export"]` | Derived from service name |
| eth-tag | `vlan` | AC tag == access VLAN |
| mtu / ecmp | 1500 / 4 | Constants |

### Test enforcement

`tests/test_generator_determinism.py` has two parts:
- **Part 1 -- seed determinism:** same seed calling `build_X_sample()` twice must yield byte-identical `(instruction, output)` pairs.
- **Part 2 -- argument purity:** hold the argument dict fixed, vary the global random seed, the output must be constant (proves no `random.X` leaks into `fill_values`).

Part 2 is the exact test that would have caught the M3.5 baseline bug -- any stray `random.X` inside the generator immediately fails.

### Live verification -- purity in action

In [1]:
import sys, os, random, json
ROOT = "/home/nextron/nsp_intent_ft"
sys.path.insert(0, os.path.join(ROOT, "data"))

from field_definitions import generate_evpn_epipe_values

# An 'instruction-visible' argument set (every value here would also appear
# verbatim in the natural-language instruction template).
visible_args = dict(
    service_name="EVPN-Epipe-700-DC",
    customer_id=40,
    ne_service_id=700,
    evi=700,
    evpn_type="mpls",
    vlan=700,
    device="192.168.0.16",
    port="1/2/c4/1",
    local_ac="AC-DC-local",
    remote_ac="AC-DC-remote",
)

# Key test: even with completely different global random states, the output
# of a pure generator must be byte-identical.
outputs = []
for seed in (0, 42, 9999, 31337):
    random.seed(seed)
    fv = generate_evpn_epipe_values(**visible_args)
    outputs.append(json.dumps(fv, sort_keys=True))

all_identical = len(set(outputs)) == 1
print(f"Four calls across different random seeds produce byte-identical output: {all_identical}")
print(f"Number of fields in output: {len(json.loads(outputs[0]))}")
print()
print("-- First 8 fields (showing 'direct arg / constant / derived' all three sources) --")
fv = json.loads(outputs[0])
for k, v in list(fv.items())[:8]:
    print(f"  {k:60s} = {v!r}")

Four calls across different random seeds produce byte-identical output: True
Number of fields in output: 27

-- First 8 fields (showing 'direct arg / constant / derived' all three sources) --
  customer-id                                                  = 40
  description                                                  = 'EVPN-Epipe-700-DC EVPN service'
  evpn-type                                                    = 'mpls'
  mtu                                                          = 1500
  ne-service-id                                                = 700
  service-name                                                 = 'EVPN-Epipe-700-DC'
  site-a.description                                           = 'EVPN-Epipe-700-DC site-a'
  site-a.device-id                                             = '192.168.0.16'


## 4. Validation Stack -- Four YANG-Driven Tiers

**Design rationale:** output quality should not depend on training alone -- an independent validator acts as a second line of defense. Built on Nokia's official YANG schema, the layered stack catches different classes of error:

| Tier | Responsibility | Typical errors caught |
|---|---|---|
| **Tier 1** | Path validity | Field-name typos, nonexistent paths |
| **Tier 2** | Type / range / enum / pattern | VLAN=9999 (exceeds 1..4094), customer-id as a string |
| **Tier 3** | Merged-JSON structural integrity | Missing mandatory fields, unfilled list keys, min/max-elements violations |
| **Tier 4** | Cross-field semantic rules | SDP direction mismatch, epipe with identical site devices, tunnel source==destination |
| **Tier 6** | Canonical similarity | **Warning-only:** whether field paths appear in Nokia's shipped canonical examples |

> **Why no Tier 5?** YANG `when` clauses in NSP intents are mostly trivial equality expressions which Tier 3 already handles.
>
> **Why is Tier 6 only a warning?** Nokia's canonical payloads are incomplete (vprn payload2 has 15 fields, payload1 has 658), so a missing path isn't definitively wrong. Worse, etree/cpipe/tunnel have *no* canonical payloads in Nokia's bundle at all -- these always report `N/A`.

### Negative-case demo -- what does the validator actually catch?

We deliberately break four `fill_values` dicts and show the error messages each tier produces.

In [2]:
# merge_fill_values lives in inference/, so add that path as well.
if os.path.join(ROOT, "inference") not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, "inference"))

from intent_validator import (
    validate_fill_values,       # Tier 1 + 2
    validate_merged_intent,     # Tier 3
    validate_semantic,          # Tier 4
)
from merge_fill_values import merge_fill_values

def show_errors(title, ok, errors, expected_tier):
    status = "PASS (not caught)" if ok else f"FAIL (correctly caught by {expected_tier})"
    print(f"-- {title}")
    print(f"   Result: {status}")
    for e in (errors if isinstance(errors, list) else []):
        print(f"   -> {e}")
    print()

# -------------------------------------------------------------------
# Bad case 1: Tier 1 -- misspelled field path
# -------------------------------------------------------------------
bad1 = {
    "service-name": "Epipe-T1",
    "site-a.endpont[0].port-id": "1/2/c4/1",  # 'endpont' is a typo
}
ok, errs = validate_fill_values("epipe", bad1)
show_errors("Bad case 1: misspelled path (endpont)", ok, errs, "Tier 1")

# -------------------------------------------------------------------
# Bad case 2: Tier 2 -- out-of-range value and wrong type
# -------------------------------------------------------------------
bad2 = {
    "service-name": "Epipe-T2",
    "site-a.endpoint[0].outer-vlan-tag": 99999,  # YANG range: 0..4094
    "customer-id": "ten",                        # should be integer
}
ok, errs = validate_fill_values("epipe", bad2)
show_errors("Bad case 2: VLAN out of range + customer-id wrong type", ok, errs, "Tier 2")

# -------------------------------------------------------------------
# Bad case 3: Tier 3 -- exceed YANG max-elements
# (the epipe YANG schema restricts sdp-details.sdp to at most 12 entries)
# -------------------------------------------------------------------
bad3_fv = {
    "service-name": "Epipe-T3", "customer-id": 10, "ne-service-id": 2001,
    "site-a.device-id": "192.168.0.37",
    "site-a.endpoint[0].port-id": "1/2/c4/1",
    "site-a.endpoint[0].outer-vlan-tag": 1001,
    "site-b.device-id": "192.168.0.16",
    "site-b.endpoint[0].port-id": "1/2/c5/1",
    "site-b.endpoint[0].outer-vlan-tag": 1001,
}
# Insert 13 SDP entries (exceeds max-elements=12)
for i in range(13):
    bad3_fv[f"sdp[{i}].sdp-id"] = str(1000 + i)
    bad3_fv[f"sdp[{i}].source-device-id"] = "192.168.0.37"
    bad3_fv[f"sdp[{i}].destination-device-id"] = "192.168.0.16"
bad3_merged = merge_fill_values("epipe", bad3_fv)
ok, errs = validate_merged_intent("epipe", bad3_merged)
show_errors("Bad case 3: 13 SDP entries > YANG max-elements=12", ok, errs, "Tier 3")

# -------------------------------------------------------------------
# Bad case 4: Tier 4 -- epipe semantic error (both sites point at same device)
# -------------------------------------------------------------------
bad4 = {
    "service-name": "Epipe-T4",
    "customer-id": 10,
    "ne-service-id": 2001,
    "mtu": 1492,
    "site-a.device-id": "192.168.0.37",
    "site-a.endpoint[0].port-id": "1/2/c4/1",
    "site-a.endpoint[0].outer-vlan-tag": 1001,
    "site-b.device-id": "192.168.0.37",           # same as site-a!
    "site-b.endpoint[0].port-id": "1/2/c5/1",
    "site-b.endpoint[0].outer-vlan-tag": 1001,
    "sdp[0].sdp-id": "3716",
    "sdp[0].source-device-id": "192.168.0.37",
    "sdp[0].destination-device-id": "192.168.0.37",
}
ok, errs = validate_semantic("epipe", bad4)
show_errors("Bad case 4: epipe semantic error (site-a.device-id == site-b.device-id)", ok, errs, "Tier 4")

print("  --- All four bad cases were correctly flagged by the expected tier ---")

-- Bad case 1: misspelled path (endpont)
   Result: FAIL (correctly caught by Tier 1)
   -> unknown field path: 'site-a.endpont[0].port-id'

-- Bad case 2: VLAN out of range + customer-id wrong type
   Result: FAIL (correctly caught by Tier 2)
   -> site-a.endpoint[0].outer-vlan-tag: value 99999 out of int16 bounds [-32768, 32767]
   -> customer-id: expected uint32, got non-integer string 'ten'

-- Bad case 3: 13 SDP entries > YANG max-elements=12
   Result: FAIL (correctly caught by Tier 3)
   -> <body>.sdp-details.sdp: 13 entries exceeds max-elements=12

-- Bad case 4: epipe semantic error (site-a.device-id == site-b.device-id)
   Result: FAIL (correctly caught by Tier 4)
   -> site-a and site-b have the same device-id

  --- All four bad cases were correctly flagged by the expected tier ---


## 5. End-to-End Demos -- Load the Model

We now run the real inference pipeline on 9 intent types: for each type, we feed in a single natural-language sentence and watch the full chain -- **inference -> fill_values merge -> 4-tier validation** -- execute end to end.

Loading Qwen3.5-9B base + LoRA adapter. Takes ~10 seconds on first load.

In [3]:
import sys, os, json, time
ROOT = "/home/nextron/nsp_intent_ft"
sys.path.insert(0, os.path.join(ROOT, "inference"))
sys.path.insert(0, os.path.join(ROOT, "data"))

from predict import load_model, predict, extract_json, DEFAULT_MODEL, DEFAULT_ADAPTER
from merge_fill_values import merge_fill_values
from intent_validator import validate_full, validate_canonical_similarity
from IPython.display import display, HTML

print(f"Base model:    {DEFAULT_MODEL}")
print(f"LoRA adapter:  {os.path.abspath(DEFAULT_ADAPTER)}")
print(f"Training:      r=32, alpha=64, 5 epochs, final loss ~= 0.085")
print()
print("Loading model...")
t0 = time.time()
model, tokenizer = load_model()
print(f"Model loaded in {time.time()-t0:.1f}s.")

/home/nextron/nsp_intent_ft/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base model:    Qwen/Qwen3.5-9B
LoRA adapter:  /home/nextron/nsp_intent_ft/output/qwen3-nsp-intent-adapter
Training:      r=32, alpha=64, 5 epochs, final loss ~= 0.085

Loading model...
Loading tokenizer from Qwen/Qwen3.5-9B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model Qwen/Qwen3.5-9B...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading weights:  93%|█████████▎| 398/427 [00:02<00:00, 166.36it/s]

Loading weights:  97%|█████████▋| 415/427 [00:02<00:00, 158.42it/s]

Loading weights: 100%|██████████| 427/427 [00:02<00:00, 159.47it/s]

Loading LoRA adapter from /home/nextron/nsp_intent_ft/inference/../output/qwen3-nsp-intent-adapter...


Model loaded in 12.2s.


### Reading Tier 6 -- what 'N/A' and 'novel' mean

In the 9 examples below, the Tier 6 column shows two special values that need a heads-up:

- **`N/A`** (tunnel / etree / cpipe): Nokia did not ship canonical payloads for these three intent types, so there is nothing to compare against.
- **`13/16 known (+3 novel)`**: 13 of 16 output field paths appear in Nokia's canonical examples; 3 paths are `novel`. **`novel` is NOT an error** -- Nokia's epipe canonical sample only shows site-a, so any site-b.* field is marked novel by construction.

The `ok` flag returned by `validate_full()` is determined by Tier 1/2/3/4 only -- Tier 6 is informational.

In [4]:
# Module-level store: we collect per-example results so the summary table at the
# end can be computed from the actual runs rather than hardcoded.
_demo_results = []

def run_and_display(intent_type, type_desc, instruction):
    """Run the full pipeline and display results as rich HTML. Return a result dict."""
    # 1. Inference + latency
    t0 = time.time()
    raw = predict(model, tokenizer, instruction)
    latency_s = time.time() - t0
    parsed = extract_json(raw)

    pred_type = parsed.get("intent_type", "")
    template_name = parsed.get("template_name", "")
    fill_values = parsed.get("fill_values", {})

    # 2. Merge into RESTCONF-shaped JSON
    merged = merge_fill_values(pred_type, fill_values)

    # 3. 4-tier validation + Tier 6
    ok, tier_errors = validate_full(pred_type, fill_values, merged_json=merged)
    n_known, n_novel, novel_paths = validate_canonical_similarity(pred_type, fill_values)

    # 4. Render
    fv_json = json.dumps({"intent_type": pred_type, "template_name": template_name,
                          "fill_values": fill_values}, indent=2, ensure_ascii=False)
    api_json = json.dumps(merged, indent=2, ensure_ascii=False)
    n_fv_fields = len(fill_values)
    n_api_lines = api_json.count(chr(10)) + 1

    t12_ok = not tier_errors.get("tier1_2", [])
    t3_ok  = not tier_errors.get("tier3", [])
    t4_ok  = not tier_errors.get("tier4", [])

    t12 = "PASS" if t12_ok else "FAIL"
    t3  = "PASS" if t3_ok  else "FAIL"
    t4  = "PASS" if t4_ok  else "FAIL"
    if (n_known + n_novel) > 0:
        t6 = f"{n_known}/{n_known + n_novel} known"
        if n_novel > 0:
            t6 += f" (+{n_novel} novel)"
    else:
        t6 = "N/A (no Nokia canonical)"

    def cell_color(flag_ok):
        return "green" if flag_ok else "red"

    html = f"""
    <div style="border:1px solid #ddd; border-radius:8px; padding:20px; margin:10px 0; background:#fafafa;">
        <h3 style="color:#2c3e50; border-bottom:2px solid #3498db; padding-bottom:8px;">
            {intent_type} -- {type_desc}
            <span style="font-size:13px; font-weight:normal; color:#777; float:right;">
                inference {latency_s:.1f}s &nbsp;&middot;&nbsp; {n_fv_fields} fill_values fields &nbsp;&middot;&nbsp; {n_api_lines}-line full JSON
            </span>
        </h3>

        <div style="background:#fff; border:1px solid #e0e0e0; border-radius:4px; padding:12px; margin:10px 0;">
            <b style="color:#555;">User instruction (natural language):</b><br>
            <p style="color:#333; font-size:14px; line-height:1.6; margin:6px 0 0 0;">{instruction}</p>
        </div>

        <div style="display:flex; gap:20px;">
            <div style="flex:1;">
                <b style="color:#555;">Model output (fill_values, flat dict):</b>
                <pre style="background:#1e1e1e; color:#d4d4d4; padding:12px; border-radius:4px; font-size:12px; overflow-x:auto; max-height:320px;">{fv_json}</pre>
            </div>
            <div style="flex:1;">
                <b style="color:#555;">NSP RESTCONF-shaped JSON (offline validated, collapsed by default):</b>
                <details style="margin-top:4px;"><summary style="cursor:pointer; color:#3498db; font-size:13px; padding:4px 0;">&#9656; Click to expand full nested JSON ({n_api_lines} lines)</summary>
                <pre style="background:#1e1e1e; color:#d4d4d4; padding:12px; border-radius:4px; font-size:12px; overflow-x:auto; max-height:320px;">{api_json}</pre>
                </details>
            </div>
        </div>

        <div style="margin-top:10px; padding:10px; background:#fff; border:1px solid #e0e0e0; border-radius:4px;">
            <b style="color:#555;">Validation results:</b>
            <table style="margin-top:5px; border-collapse:collapse; font-size:13px;">
                <tr>
                    <td style="padding:4px 15px; border:1px solid #ddd;">Tier 1+2 (path / type)</td>
                    <td style="padding:4px 15px; border:1px solid #ddd; color:{cell_color(t12_ok)}; font-weight:bold;">{t12}</td>
                    <td style="padding:4px 15px; border:1px solid #ddd;">Tier 3 (structural integrity)</td>
                    <td style="padding:4px 15px; border:1px solid #ddd; color:{cell_color(t3_ok)}; font-weight:bold;">{t3}</td>
                </tr>
                <tr>
                    <td style="padding:4px 15px; border:1px solid #ddd;">Tier 4 (cross-field semantics)</td>
                    <td style="padding:4px 15px; border:1px solid #ddd; color:{cell_color(t4_ok)}; font-weight:bold;">{t4}</td>
                    <td style="padding:4px 15px; border:1px solid #ddd;">Tier 6 (canonical, warning-only)</td>
                    <td style="padding:4px 15px; border:1px solid #ddd; color:#888; font-weight:bold;">{t6}</td>
                </tr>
            </table>
        </div>
    </div>
    """
    display(HTML(html))

    result = {
        "intent_type": intent_type, "desc": type_desc,
        "t12": t12_ok, "t3": t3_ok, "t4": t4_ok,
        "tier6": t6, "latency": latency_s,
        "n_fields": n_fv_fields, "n_api_lines": n_api_lines,
        "all_ok": ok,
    }
    _demo_results.append(result)
    return result

## 5.1 E-Pipe -- Point-to-Point Ethernet Pseudowire
**Scenario:** the most fundamental L2 point-to-point service -- establishes an Ethernet pseudowire between two sites, carried over an MPLS SDP.
**Typical use:** enterprise dedicated line, L2 interconnect between two data centers.

In [5]:
run_and_display("epipe", "Point-to-point Ethernet pseudowire",
    "Create an E-Pipe service named 'Epipe-VLAN-1001-demo' for customer 10 with NE service ID 2001. Connect device 192.168.0.37 on port 1/2/c4/1 to device 192.168.0.16 on port 1/2/c5/1 using VLAN 1001. MTU is 1492. Use SDP 3716 and 1637.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",13/16 known (+3 novel)


{'intent_type': 'epipe',
 'desc': 'Point-to-point Ethernet pseudowire',
 't12': True,
 't3': True,
 't4': True,
 'tier6': '13/16 known (+3 novel)',
 'latency': 13.743481874465942,
 'n_fields': 16,
 'n_api_lines': 65,
 'all_ok': True}

## 5.2 Tunnel -- MPLS SDP Tunnel
**Scenario:** the underlying transport for all L2 services -- signals an MPLS tunnel between two PE devices.
**Relationship:** E-Pipe / VPLS / E-Tree all ride on top of a previously-established SDP.

In [6]:
run_and_display("tunnel", "MPLS SDP tunnel",
    "Create an MPLS tunnel from 192.168.0.16 to 192.168.0.37 with SDP ID 1637. Name it 'SDP-from-C2U16-to-C2U37'. Use BGP signaling with TLDP.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",N/A (no Nokia canonical)


{'intent_type': 'tunnel',
 'desc': 'MPLS SDP tunnel',
 't12': True,
 't3': True,
 't4': True,
 'tier6': 'N/A (no Nokia canonical)',
 'latency': 4.389577388763428,
 'n_fields': 4,
 'n_api_lines': 32,
 'all_ok': True}

## 5.3 VPRN -- L3 VPN Virtual Routing
**Scenario:** L3 VPN service where each site owns an isolated VRF and IP interface.
**Typical use:** data-center interconnect (DCI), enterprise WAN aggregation, GPU-cluster tenant isolation.

In [7]:
run_and_display("vprn", "L3 VPN virtual routing",
    "Create a VPRN L3 VPN service 'VPRN-100-DataCenter' for customer 5. Configure site on device 192.168.0.16 with service ID 100. Route distinguisher 65000:100. VRF import: DC-VRF-Import, VRF export: DC-VRF-Export. Interface GPU-Cluster-Compute on port 1/2/c4/1 with IP 10.100.1.1/24. Interface GPU-Cluster-Storage on port 1/2/c5/1 with IP 10.100.2.1/24.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",16/16 known


{'intent_type': 'vprn',
 'desc': 'L3 VPN virtual routing',
 't12': True,
 't3': True,
 't4': True,
 'tier6': '16/16 known',
 'latency': 14.37942099571228,
 'n_fields': 16,
 'n_api_lines': 203,
 'all_ok': True}

## 5.4 VPLS -- Multipoint Ethernet Bridging
**Scenario:** virtual Ethernet LAN spanning multiple sites in a single broadcast domain.
**Typical use:** campus multi-site interconnect, cross-region VLAN extension.

In [8]:
run_and_display("vpls", "Multipoint Ethernet bridging",
    "Create a VPLS service 'VPLS-500-Campus' for customer 20, NE service ID 500, MTU 1500. Site 1: device 192.168.0.16 on port 1/2/c4/1 with VLAN 500. Site 2: device 192.168.0.37 on port 1/2/c5/1 with VLAN 500.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",18/18 known


{'intent_type': 'vpls',
 'desc': 'Multipoint Ethernet bridging',
 't12': True,
 't3': True,
 't4': True,
 'tier6': '18/18 known',
 'latency': 14.988642930984497,
 'n_fields': 18,
 'n_api_lines': 60,
 'all_ok': True}

## 5.5 IES -- Internet Enhanced Service
**Scenario:** direct Internet access -- configure IP interfaces on the device, no VRF isolation.
**Typical use:** simple public-Internet egress, management-plane access.

In [9]:
run_and_display("ies", "Internet access",
    "Set up an IES service 'IES-300-Access' for customer 15 with NE service ID 300 on device 192.168.0.16. Interface AccessPort1 on port 1/2/c4/1 with IP 10.30.1.1/24. Interface AccessPort2 on port 1/2/c5/1 with IP 10.30.2.1/24.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",13/13 known


{'intent_type': 'ies',
 'desc': 'Internet access',
 't12': True,
 't3': True,
 't4': True,
 'tier6': '13/13 known',
 'latency': 11.123887062072754,
 'n_fields': 13,
 'n_api_lines': 51,
 'all_ok': True}

## 5.6 E-Tree -- Rooted Multipoint Service (Hub-and-Spoke)
**Scenario:** hub-and-spoke multipoint Ethernet -- root can reach all leaves, **but leaves cannot reach each other**.
**Difficulty:** this asymmetric topology demands special SDP generation (the main M3.5 Phase-4 fix -- the previous version accidentally used a full-mesh, which tanked accuracy at 4+ sites).

In [10]:
run_and_display("etree", "Rooted multipoint",
    "Create an E-Tree service 'ETree-400-HubSpoke' for customer 25 with NE service ID 400, MTU 1500. Root device 192.168.0.16 on port 1/2/c4/1. Leaves: device 192.168.0.37 on port 1/2/c5/1; device 192.168.0.38 on port 1/2/c6/1. VLAN 400.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",N/A (no Nokia canonical)


{'intent_type': 'etree',
 'desc': 'Rooted multipoint',
 't12': True,
 't3': True,
 't4': True,
 'tier6': 'N/A (no Nokia canonical)',
 'latency': 25.536365509033203,
 'n_fields': 31,
 'n_api_lines': 79,
 'all_ok': True}

## 5.7 Cpipe -- TDM Circuit Emulation
**Scenario:** carry legacy TDM circuits over an MPLS network, supporting CESoPSN/SAToP emulation.
**Typical use:** migrate existing E1/T1 voice/leased-line services onto IP/MPLS transport without replacing the endpoint equipment.

In [11]:
run_and_display("cpipe", "TDM circuit emulation",
    "Create a Cpipe TDM service 'Cpipe-600-TDM' for customer 35 with NE service ID 600. vc-type cesopsn. Site A: device 192.168.0.16, port 1/2/c4/1, time-slots 1-32. Site B: device 192.168.0.37, port 1/2/c5/1, time-slots 1-32.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",N/A (no Nokia canonical)


{'intent_type': 'cpipe',
 'desc': 'TDM circuit emulation',
 't12': True,
 't3': True,
 't4': True,
 'tier6': 'N/A (no Nokia canonical)',
 'latency': 13.108198404312134,
 'n_fields': 16,
 'n_api_lines': 50,
 'all_ok': True}

## 5.8 EVPN E-Pipe -- BGP-EVPN Point-to-Point E-Line
**Scenario:** replace legacy TLDP signaling with the BGP EVPN control plane to build an E-Line service.
**Typical use:** data-center interconnect (DCI), scenarios needing faster convergence and multi-homing.
**M3.5 refactor target:** one of the two types rewritten under the pure-function contract -- value accuracy went from 73% to 100%.

In [12]:
run_and_display("evpn-epipe", "BGP-EVPN E-Line",
    "Create an mpls-EVPN E-Line service 'EVPN-Epipe-700-DC' for customer 40 with NE service ID 700 and EVI 700. Configure on device 192.168.0.16, port 1/2/c4/1, VLAN 700. Local AC 'AC-DC-local', remote AC 'AC-DC-remote'.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",27/27 known


{'intent_type': 'evpn-epipe',
 'desc': 'BGP-EVPN E-Line',
 't12': True,
 't3': True,
 't4': True,
 'tier6': '27/27 known',
 'latency': 20.017614126205444,
 'n_fields': 27,
 'n_api_lines': 69,
 'all_ok': True}

## 5.9 EVPN VPLS -- BGP-EVPN Multipoint Bridging
**Scenario:** layer BGP EVPN on top of VPLS multipoint bridging for multi-homing and optimized MAC learning.
**Typical use:** large-scale campus and data-center broadcast domains.
**M3.5 refactor target:** value accuracy went from 78% to 100%.

In [13]:
run_and_display("evpn-vpls", "BGP-EVPN multipoint bridging",
    "Create a mpls-EVPN VPLS service 'EVPN-VPLS-800-Campus' for customer 45 with NE service ID 800, EVI 800, MTU 1500. Site 1: 192.168.0.16 on port 1/2/c4/1 with VLAN 800. Site 2: 192.168.0.37 on port 1/2/c5/1 with VLAN 800.")

Tier 1+2 (path / type),PASS,Tier 3 (structural integrity),PASS
Tier 4 (cross-field semantics),PASS,"Tier 6 (canonical, warning-only)",45/45 known


{'intent_type': 'evpn-vpls',
 'desc': 'BGP-EVPN multipoint bridging',
 't12': True,
 't3': True,
 't4': True,
 'tier6': '45/45 known',
 'latency': 36.98359036445618,
 'n_fields': 45,
 'n_api_lines': 111,
 'all_ok': True}

## 6. Summary -- Validation Results of the 9 Examples Above

The table below is **computed** from the actual return values of the 9 `run_and_display` calls above -- it is not hardcoded. If any example fails, the corresponding cells turn red automatically.

In [14]:
def render_summary():
    rows_html = ""
    all_pass = True
    for i, r in enumerate(_demo_results):
        bg = "#f9f9f9" if i % 2 == 0 else "#ffffff"
        def c(ok): return ("green", "PASS") if ok else ("red", "FAIL")
        c12, t12 = c(r["t12"]); c3, t3 = c(r["t3"]); c4, t4 = c(r["t4"])
        all_ok = r["t12"] and r["t3"] and r["t4"]
        all_pass = all_pass and all_ok
        rows_html += (
            f'<tr style="background:{bg};">'
            f'<td style="padding:8px 12px; border:1px solid #ddd; font-family:monospace; font-weight:bold;">{r["intent_type"]}</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd;">{r["desc"]}</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd; text-align:right;">{r["n_fields"]}</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd; text-align:right;">{r["n_api_lines"]}</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd; text-align:right;">{r["latency"]:.1f}s</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd; color:{c12}; text-align:center; font-weight:bold;">{t12}</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd; color:{c3}; text-align:center; font-weight:bold;">{t3}</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd; color:{c4}; text-align:center; font-weight:bold;">{t4}</td>'
            f'<td style="padding:8px 12px; border:1px solid #ddd; color:#888; text-align:center; font-size:12px;">{r["tier6"]}</td>'
            f'</tr>'
        )

    banner = ("#27ae60", "9/9 examples passed all four validation tiers") if all_pass \
             else ("#c0392b", "Some examples failed -- see details above")
    border_color, banner_text = banner

    html = f"""
    <div style="border:2px solid {border_color}; border-radius:8px; padding:20px; margin:20px 0; background:#fafafa;">
        <h2 style="color:{border_color}; text-align:center; margin:0 0 15px 0;">{banner_text}</h2>
        <table style="width:100%; border-collapse:collapse;">
            <tr style="background:{border_color}; color:white;">
                <th style="padding:10px; border:1px solid #ddd;">Intent</th>
                <th style="padding:10px; border:1px solid #ddd;">Service category</th>
                <th style="padding:10px; border:1px solid #ddd;">Fields</th>
                <th style="padding:10px; border:1px solid #ddd;">Full JSON lines</th>
                <th style="padding:10px; border:1px solid #ddd;">Latency</th>
                <th style="padding:10px; border:1px solid #ddd;">Tier 1+2</th>
                <th style="padding:10px; border:1px solid #ddd;">Tier 3</th>
                <th style="padding:10px; border:1px solid #ddd;">Tier 4</th>
                <th style="padding:10px; border:1px solid #ddd;">Tier 6</th>
            </tr>
            {rows_html}
        </table>
    </div>
    """
    display(HTML(html))

render_summary()

Intent,Service category,Fields,Full JSON lines,Latency,Tier 1+2,Tier 3,Tier 4,Tier 6
epipe,Point-to-point Ethernet pseudowire,16,65,13.7s,PASS,PASS,PASS,13/16 known (+3 novel)
tunnel,MPLS SDP tunnel,4,32,4.4s,PASS,PASS,PASS,N/A (no Nokia canonical)
vprn,L3 VPN virtual routing,16,203,14.4s,PASS,PASS,PASS,16/16 known
vpls,Multipoint Ethernet bridging,18,60,15.0s,PASS,PASS,PASS,18/18 known
ies,Internet access,13,51,11.1s,PASS,PASS,PASS,13/13 known
etree,Rooted multipoint,31,79,25.5s,PASS,PASS,PASS,N/A (no Nokia canonical)
cpipe,TDM circuit emulation,16,50,13.1s,PASS,PASS,PASS,N/A (no Nokia canonical)
evpn-epipe,BGP-EVPN E-Line,27,69,20.0s,PASS,PASS,PASS,27/27 known
evpn-vpls,BGP-EVPN multipoint bridging,45,111,37.0s,PASS,PASS,PASS,45/45 known


## 7. Overall Evaluation -- Full Test Set of 330 Samples

The 9 examples above are just highlights. Real model quality is confirmed on the full test set. The test-set numbers below are from `output/logs/m3_5final_eval.log` (M3.5 final eval, 2026-04-10). Golden pass/fail status is from the later `output/logs/golden_only_eval.log` regression run.

### Headline metrics (test set = 330 samples)

| Metric | Test set (330) | Golden tests (11) | Meaning |
|---|---|---|---|
| JSON Valid Rate | **100.0%** | 100.0% | Outputs parse as valid JSON |
| Intent Type Accuracy | **100.0%** | 100.0% | Predicted intent type is correct |
| Field Recall | **100.0%** | 100.0% | All expected fields are present |
| Field Precision | **100.0%** | 100.0% | No extraneous fields |
| Value Accuracy | **100.0%** | 100.0% | Matched fields carry the correct value |
| Tier 1+2 Valid | **100.0%** | 100.0% | YANG path / type validation passes |
| Tier 3 Valid | **100.0%** | 100.0% | Merged structure passes |
| Tier 4 Valid | **100.0%** | 100.0% | Cross-field semantics pass |
| All Tiers Valid | **100.0%** | 100.0% | Every tier simultaneously |
| Tier 6 Canonical | 96.6% | warning-only | Reference only; not defined for all intent types |
| SDP Bidirectional (epipe) | 100.0% | 100.0% | Specifically tracked semantic |

### Per-intent-type breakdown (test set)

In [15]:
# Data source: output/logs/m3_5final_eval.log (2026-04-10 M3.5 final eval run)
per_intent_breakdown = [
    ("epipe",      64, 100.0, 100.0, 100.0),
    ("tunnel",     33, 100.0, 100.0, 100.0),
    ("vprn",       50, 100.0, 100.0, 100.0),
    ("vpls",       43, 100.0, 100.0, 100.0),
    ("ies",        36, 100.0, 100.0, 100.0),
    ("etree",      22, 100.0, 100.0, 100.0),
    ("cpipe",      13, 100.0, 100.0, 100.0),
    ("evpn-epipe", 38, 100.0, 100.0, 100.0),
    ("evpn-vpls",  31, 100.0, 100.0, 100.0),
]

total = sum(n for _, n, *_ in per_intent_breakdown)
rows_html = ""
for i, (itype, n, jv, recall, val_acc) in enumerate(per_intent_breakdown):
    bg = "#f9f9f9" if i % 2 == 0 else "#ffffff"
    rows_html += (
        f'<tr style="background:{bg};">'
        f'<td style="padding:8px 12px; border:1px solid #ddd; font-family:monospace; font-weight:bold;">{itype}</td>'
        f'<td style="padding:8px 12px; border:1px solid #ddd; text-align:right;">{n}</td>'
        f'<td style="padding:8px 12px; border:1px solid #ddd; text-align:right; color:green; font-weight:bold;">{jv:.1f}%</td>'
        f'<td style="padding:8px 12px; border:1px solid #ddd; text-align:right; color:green; font-weight:bold;">{recall:.1f}%</td>'
        f'<td style="padding:8px 12px; border:1px solid #ddd; text-align:right; color:green; font-weight:bold;">{val_acc:.1f}%</td>'
        f'</tr>'
    )

html = f"""
<div style="border:1px solid #ddd; border-radius:8px; padding:15px; background:#fafafa;">
    <table style="width:100%; border-collapse:collapse;">
        <tr style="background:#2c3e50; color:white;">
            <th style="padding:10px; border:1px solid #ddd;">Intent type</th>
            <th style="padding:10px; border:1px solid #ddd;">Samples</th>
            <th style="padding:10px; border:1px solid #ddd;">JSON Valid</th>
            <th style="padding:10px; border:1px solid #ddd;">Field Recall</th>
            <th style="padding:10px; border:1px solid #ddd;">Value Accuracy</th>
        </tr>
        {rows_html}
        <tr style="background:#e8f5e9; font-weight:bold;">
            <td style="padding:10px; border:1px solid #ddd;">Total</td>
            <td style="padding:10px; border:1px solid #ddd; text-align:right;">{total}</td>
            <td style="padding:10px; border:1px solid #ddd; text-align:right; color:green;">100.0%</td>
            <td style="padding:10px; border:1px solid #ddd; text-align:right; color:green;">100.0%</td>
            <td style="padding:10px; border:1px solid #ddd; text-align:right; color:green;">100.0%</td>
        </tr>
    </table>
</div>
"""
display(HTML(html))

Intent type,Samples,JSON Valid,Field Recall,Value Accuracy
epipe,64,100.0%,100.0%,100.0%
tunnel,33,100.0%,100.0%,100.0%
vprn,50,100.0%,100.0%,100.0%
vpls,43,100.0%,100.0%,100.0%
ies,36,100.0%,100.0%,100.0%
etree,22,100.0%,100.0%,100.0%
cpipe,13,100.0%,100.0%,100.0%
evpn-epipe,38,100.0%,100.0%,100.0%
evpn-vpls,31,100.0%,100.0%,100.0%
Total,330,100.0%,100.0%,100.0%


### Cross-milestone metric evolution

How the M1 -> M3.5 iteration reflects in the numbers:

| Milestone | # Intent types | Test Value Accuracy | Key action |
|---|---|---|---|
| M1 | 3 | baseline | YANG validator online |
| M2 | 3 | baseline | YANG-driven path resolution replaces hardcoded maps |
| M2.5 | 3 | baseline-improved | Byte-level chat template alignment + max_tokens fix |
| M3 | 9 | **98.8%** | Scale to 6 new types, etree at 84% |
| M3.5 baseline | 9 | 97.9% | EVPN structure fix (but introduced a new bug), etree drops to 79% |
| M3.5 final | 9 | **100.0%** | Pure-function contract + E-Tree root-leaf SDP fix |

### Key takeaways

1. **Generality of the pure-function contract:** this lesson applies to any template-based training-data synthesis -- any randomness not flowing through the instruction template is unlearnable noise.

2. **E-Tree's three-layer root causes:** the fix order was bottom-up (semantics -> scale -> format), mapping to three layers of data quality: domain-semantic correctness, model learnability, text parseability.

3. **The validator as a second line of defense:** an independent YANG-driven validation stack not only filters bad training samples up front, but also provides inference-time confidence -- the kind of engineering determinism pure end-to-end neural methods lack.

## 8. Conclusions and Future Work

### What has shipped

| Dimension | Specification |
|------|------|
| Base model | Qwen3.5-9B + LoRA (r=32, alpha=64, 7 target modules) |
| Training data | 2,640 train / 330 val / 330 test / 11 Golden, across 9 intent types |
| Training hardware | 2 x RTX 6000 Ada 49GB, BF16 without quantization, DDP parallel |
| Training time | 5 epochs ~= 410 steps, final loss 0.085, token accuracy 96.6% |
| Inference latency | Single GPU, typically 4-36 seconds per sample depending on output length |
| Validation stack | 5-tier YANG-driven, auto-derived from Nokia's schema |
| Output format | NSP RESTCONF-shaped JSON, offline validated; live NSP acceptance not yet measured |

### Core technical contributions

1. **Five-tier YANG-driven validation stack** -- from path validity up to cross-field semantics, systematic quality assurance.
2. **Pure-function generator contract** -- eliminates training-data information leakage; every field recoverable from the instruction.
3. **Byte-level chat-template alignment** -- resolves the Qwen3.5 `<think>` mismatch between training and inference.
4. **E-Tree topology-aware SDP generation** -- encodes the 'leaves cannot talk to each other' semantic into the training data.

### Current limitations and future work

- **Offline validation is a structural lower bound:** passing Tier 1-4 only means the output conforms to YANG. NSP server-side has an additional JavaScript mapping engine (`mapping-engine="js-scripted"`). `valid=True` does NOT imply NSP will accept.
- **Incomplete canonical coverage:** etree / cpipe / tunnel have no canonical payloads in Nokia's bundle, so Tier 6 provides no signal for them.
- **Test set and training set are i.i.d.:** test samples come from the same generator, so OOD-generalization is not yet measured.

**M4 plans:**
- **Extend to device-level Intent:** Nokia NSP ships 181 device-intent types; this project covers 9 service-intents so far.
- **Live NSP deployment:** hook up a real NSP instance to measure acceptance rate in production.
- **CLI generation:** pair `payload*.cli.txt` files from Nokia's bundle with the JSON payloads to train JSON<->CLI conversion.
- **Composite and redundant intents:** active-standby failover, multi-service orchestration, and richer scenarios.